In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_account_move_line AS
SELECT
    line.move_line_id,
    move.move_sk,
    CAST(date_format(line.date, 'yyyyMMdd') AS INT) AS date_sk,
    CAST(date_format(line.date, 'yyyyMM') AS INT) AS period_sk,
    account.account_sk,
    journal.journal_sk,
    customer.customer_sk,
    line.amount_currency,
    line.amount_residual,
    line.balance,
    line.credit,
    line.debit,
    line.ref,
    line.created_at,
    line.updated_at,
    line.is_updated,
    line.is_deleted
FROM bakery_pos_develop.silver.account_move_line line
INNER JOIN
bakery_pos_develop.gold.dim_account_move move
ON
line.move_id = move.move_id
INNER JOIN
bakery_pos_develop.gold.dim_account_account account
ON
line.account_id = account.account_id
INNER JOIN
bakery_pos_develop.gold.dim_account_journal journal
ON
line.journal_id = journal.journal_id
INNER JOIN
bakery_pos_develop.gold.dim_res_partner customer
ON
line.partner_id = customer.customer_id
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY line.move_line_id
    ORDER BY line.updated_at DESC
) =1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.fact_account_move_line (
    move_line_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    move_line_id BIGINT,
    move_sk BIGINT,
    date_sk INT,
    period_sk INT,
    account_sk BIGINT,
    journal_sk BIGINT,
    customer_sk BIGINT,
    amount_currency DECIMAL(16,2),
    amount_residual DECIMAL(16,2),
    balance FLOAT,
    credit FLOAT,
    debit FLOAT,
    ref STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (move_line_sk),
    FOREIGN KEY (move_sk) REFERENCES bakery_pos_develop.gold.dim_account_move(move_sk),
    FOREIGN KEY (date_sk) REFERENCES bakery_pos_develop.gold.dim_date(date_sk),
    FOREIGN KEY (period_sk) REFERENCES bakery_pos_develop.gold.dim_period(period_sk),
    FOREIGN KEY (account_sk) REFERENCES bakery_pos_develop.gold.dim_account_account(account_sk),
    FOREIGN KEY (journal_sk) REFERENCES bakery_pos_develop.gold.dim_account_journal(journal_sk),
    FOREIGN KEY (customer_sk) REFERENCES bakery_pos_develop.gold.dim_res_partner(customer_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.fact_account_move_line AS tgt
USING stg_account_move_line AS src
ON tgt.move_line_id = src.move_line_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.move_sk = src.move_sk,
        tgt.date_sk = src.date_sk,
        tgt.period_sk = src.period_sk,
        tgt.account_sk = src.account_sk,
        tgt.journal_sk = src.journal_sk,
        tgt.customer_sk = src.customer_sk,
        tgt.amount_currency = src.amount_currency,
        tgt.amount_residual = src.amount_residual,
        tgt.balance = src.balance,
        tgt.credit = src.credit,
        tgt.debit = src.debit,
        tgt.ref = src.ref,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        move_line_id,
        move_sk,
        date_sk,
        period_sk,
        account_sk,
        journal_sk,
        customer_sk,
        amount_currency,
        amount_residual,
        balance,
        credit,
        debit,
        ref,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.move_line_id,
        src.move_sk,
        src.date_sk,
        src.period_sk,
        src.account_sk,
        src.journal_sk,
        src.customer_sk,
        src.amount_currency,
        src.amount_residual,
        src.balance,
        src.credit,
        src.debit,
        src.ref,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )